# Nested feature selection — XGBoost

Leakage-free feature selection for **xgboost** on the 2-class ground-type problem, using the SAME
nested forward-selection harness as the k-NN / SVM notebooks so all five are directly comparable.
Forward selection runs inside each outer leave-user-out fold (features chosen on an inner grouped CV
of that fold's training users only), so the reported macro-F1 has no selection bias.

XGBoost is scale-INVARIANT and robust to redundant features, so — unlike k-NN/SVM — it won't be hurt by carrying extra features. Nested SFS here gives the leakage-free accuracy and the features that actually drive cross-user generalization, to compare against your manual 36-set.

*Honest limit:* 2-class CV rests on 3 users (soft/hard has 4 users) -> large fold variance; treat gaps under
~3 macro-F1 points as noise. Classifier is UNTUNED (matches the other notebooks' default settings).

In [1]:
import numpy as np, pandas as pd, warnings, time
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# ================= CONFIG =================
DATA_DIR='..'
MAX_K=20                       # SFS cap (selection plateaus early)
SFS_TOL=0.0005
SUBSAMPLE_PER_CLASS=None        # XGBoost is cheap; use all data
SFS_JOBS=4                      # thermal-safe parallelism (half of 8 cores)
CLASS_WEIGHT_STRENGTH=0.5
RANDOM_STATE=42

DROP={'total_p_mean','total_pmax','impact_mag','impact_mag_norm','heel_impulse','heel_mean_ratio',
      'fore_mean_ratio','heel_pmax_ratio','fore_pmax_ratio'}|{f'pressure_{i:02d}_mean_ratio' for i in range(1,13)}
META={'foot','step_id','surface','source_user','source_recording'}

dfs=[]
for fn,lab in [('soft_dry.csv','soft'),('hard_dry.csv','hard')]:
    d=pd.read_csv(f'{DATA_DIR}/{fn}'); d['surface']=lab; dfs.append(d)
df=pd.concat(dfs,ignore_index=True)
FEATS=[c for c in df.columns if c not in META and c not in DROP]
classes=sorted(df.surface.unique()); y=df.surface.values; grp=df.source_user.values
_users_per_class=df.groupby('surface')['source_user'].nunique()
N_SPLITS=int(_users_per_class.min())
_le=LabelEncoder().fit(classes)
print(f'{len(df)} steps | {len(FEATS)} features | users/class {_users_per_class.to_dict()} -> N_SPLITS={N_SPLITS}')
print(f'classifier: xgboost | SUBSAMPLE_PER_CLASS={SUBSAMPLE_PER_CLASS} | SFS_JOBS={SFS_JOBS}')

16270 steps | 49 features | users/class {'hard': 4, 'soft': 4} -> N_SPLITS=4
classifier: xgboost | SUBSAMPLE_PER_CLASS=None | SFS_JOBS=4


In [2]:
def _sw(ytr):
    cs,cnt=np.unique(ytr,return_counts=True); n,k=len(ytr),len(cs)
    w={c:(n/(k*ct))**CLASS_WEIGHT_STRENGTH for c,ct in zip(cs,cnt)}
    return np.array([w[v] for v in ytr])

def make_clf():
    return XGBClassifier(max_depth=4, n_estimators=250, learning_rate=0.05, subsample=0.9,
                         colsample_bytree=0.8, eval_metric='mlogloss', random_state=RANDOM_STATE, verbosity=0)

def _subsample(idx, yv, per_class, seed=RANDOM_STATE):
    if not per_class: return idx
    rng=np.random.default_rng(seed); keep=[]
    for c in np.unique(yv[idx]):
        ci=idx[yv[idx]==c]; keep+=list(rng.choice(ci,min(len(ci),per_class),replace=False))
    return np.array(sorted(keep))

Xall=df[FEATS].fillna(0).values

def fit_score(cols, tr, te, yv):
    tr=_subsample(tr, yv, SUBSAMPLE_PER_CLASS)
    m=make_clf(); m.fit(Xall[np.ix_(tr,cols)], _le.transform(yv[tr]), sample_weight=_sw(yv[tr]))
    pred=_le.inverse_transform(m.predict(Xall[np.ix_(te,cols)]))
    return f1_score(yv[te], pred, labels=classes, average='macro')

def inner_cv_score(cols, tr, yv, gv):
    inner=StratifiedGroupKFold(n_splits=max(2,min(N_SPLITS-1,len(np.unique(gv)))),shuffle=True,random_state=0)
    sc=[]
    for itr,ite in inner.split(Xall[tr][:,cols], yv[tr], gv):
        if len(np.unique(yv[tr][itr]))<len(classes): continue
        sc.append(fit_score(cols, tr[itr], tr[ite], yv))
    return np.mean(sc) if sc else -1

In [3]:
# ===== nested forward selection with LIVE progress =====
splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)
_folds=[(tr,te) for tr,te in splitter.split(Xall,y,grp) if len(np.unique(y[tr]))==len(classes)]
print(f'nested forward selection for xgboost: {len(_folds)} outer folds, MAX_K={MAX_K}')
outer_mf1=[]; sel_counts={}; per_fold_sets=[]
for _fi,(tr,te) in enumerate(_folds,1):
    t0=time.time(); held=np.unique(grp[te])
    print(f'  fold {_fi}/{len(_folds)} (hold out {held}) — forward-selecting...', flush=True)
    yv,gv=y, grp[tr]
    chosen=[]; remaining=list(range(len(FEATS))); best=-1; step=0
    while remaining and len(chosen)<MAX_K:
        scores=Parallel(n_jobs=SFS_JOBS,prefer='processes')(
            delayed(inner_cv_score)(chosen+[c], tr, yv, gv) for c in remaining)
        s_best,c_best=max(zip(scores,remaining))
        if s_best-best<SFS_TOL: break
        chosen.append(c_best); best=s_best; remaining.remove(c_best); step+=1
        print(f'      + {FEATS[c_best]}  (inner MacroF1 {best*100:.1f}%, {step} feats, {time.time()-t0:.0f}s)', flush=True)
    pf=fit_score(chosen, tr, te, y)
    outer_mf1.append(pf); per_fold_sets.append([FEATS[c] for c in chosen])
    for c in chosen: sel_counts[FEATS[c]]=sel_counts.get(FEATS[c],0)+1
    print(f'    -> fold done: {len(chosen)} feats, held-out MacroF1 {pf*100:.1f}%  ({time.time()-t0:.0f}s)\n', flush=True)
outer_mf1=np.array(outer_mf1)
print(f'=== xgboost: nested (leakage-free) grouped MacroF1 {outer_mf1.mean()*100:.1f}% '
      f'(sd {outer_mf1.std()*100:.1f}) | per-fold {np.round(outer_mf1*100,1)} ===')

nested forward selection for xgboost: 4 outer folds, MAX_K=20
  fold 1/4 (hold out ['Sam']) — forward-selecting...
      + orient_rest_a_roll3_std  (inner MacroF1 68.9%, 1 feats, 5s)
      + accel_spec_entropy  (inner MacroF1 76.3%, 2 feats, 9s)
      + heel_fore_ratio  (inner MacroF1 77.6%, 3 feats, 13s)
      + rise_slope  (inner MacroF1 78.1%, 4 feats, 17s)
      + orient_rest_b_roll3_std  (inner MacroF1 78.7%, 5 feats, 21s)
      + midstance_diff  (inner MacroF1 79.0%, 6 feats, 25s)
      + time_to_peak_frac  (inner MacroF1 79.4%, 7 feats, 29s)
    -> fold done: 7 feats, held-out MacroF1 63.7%  (34s)

  fold 2/4 (hold out ['Kristian']) — forward-selecting...
      + orient_rest_a_roll3_std  (inner MacroF1 70.2%, 1 feats, 4s)
      + accel_spec_centroid  (inner MacroF1 74.5%, 2 feats, 8s)
      + orient_range_b  (inner MacroF1 75.9%, 3 feats, 13s)
      + gu_stance_frac  (inner MacroF1 76.1%, 4 feats, 17s)
    -> fold done: 4 feats, held-out MacroF1 68.4%  (22s)

  fold 3/4 (hold ou

In [4]:
# ===== selection frequency + comparison to your manual 36-set =====
freq=sorted(sel_counts.items(), key=lambda kv:-kv[1])
print(f'Feature selection frequency for xgboost (across {len(_folds)} folds):')
for f,c in freq: print(f'  {c}/{len(_folds)}  {f}')
maj=[f for f,c in freq if c> len(_folds)//2]
print(f'\nStable set (selected in > half of folds): {maj}')
print('\nCompare against your MANUAL 36-set: paste it below to see how nested selection\'s'
      ' honest number relates to the manual set on the SAME folds.')

MANUAL_36 = []   # <-- paste your 36-feature list to compare
if MANUAL_36:
    def grouped_cv(cols):
        oof=np.empty(len(y),object)
        for tr,te in _folds:
            m=make_clf(); m.fit(Xall[np.ix_(tr,[FEATS.index(c) for c in cols])],_le.transform(y[tr]),sample_weight=_sw(y[tr]))
            oof[te]=_le.inverse_transform(m.predict(Xall[np.ix_(te,[FEATS.index(c) for c in cols])]))
        fill=oof!=None
        return f1_score(y[fill],oof[fill].astype(str),labels=classes,average='macro')
    m36=grouped_cv(MANUAL_36)
    print(f'\nmanual-36 grouped MacroF1: {m36*100:.1f}%  vs  nested (leakage-free): {outer_mf1.mean()*100:.1f}%')
    d=(m36-outer_mf1.mean())*100
    print(f'   difference {d:+.1f} pts -> {"likely selection bias in manual-36" if d>outer_mf1.std()*100 else "within noise; manual-36 trustworthy"}')

Feature selection frequency for xgboost (across 4 folds):
  4/4  orient_rest_a_roll3_std
  2/4  orient_rest_b_roll3_std
  2/4  accel_spec_centroid
  2/4  orient_range_b
  1/4  accel_spec_entropy
  1/4  heel_fore_ratio
  1/4  rise_slope
  1/4  midstance_diff
  1/4  time_to_peak_frac
  1/4  gu_stance_frac
  1/4  accel_jerk_ratio
  1/4  gu_push_peak
  1/4  accel_hf_frac
  1/4  gu_active_sensors

Stable set (selected in > half of folds): ['orient_rest_a_roll3_std']

Compare against your MANUAL 36-set: paste it below to see how nested selection's honest number relates to the manual set on the SAME folds.


## How to read this
- **Nested MacroF1** is xgboost's honest, leakage-free cross-user accuracy — report this as the number.
- **Stable set** = features selected in a majority of folds; the trustworthy takeaway (single-fold picks vary with only 3 users).
- If your **manual-36 ≈ nested** (within fold sd), the manual set is trustworthy but not meaningfully better — the signal is low-dimensional. If manual-36 >> nested, the extra edge was selection bias.


## Full-48 baseline vs nested selection

The decisive check (the one missing from the 3-class notebooks): does selection actually beat using **all 48 features**? In 3-class, full-48 won on every metric. This cell reports full-48 under the same grouped CV so you can see immediately whether selection is worthwhile in 2-class.

In [ ]:
# ===== full-48 baseline (the reference nested selection must beat to be worthwhile) =====
# LESSON FROM 3-CLASS: selection looked fine until compared to the FULL feature set, which won.
# Always report this next to the nested number.
from sklearn.metrics import accuracy_score as _acc
def _full_grouped_cv():
    splitter=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)
    cols=list(range(len(FEATS))); oof=np.empty(len(y),object); pf=[]
    for tr,te in splitter.split(Xall,y,grp):
        if len(np.unique(y[tr]))<len(classes): continue
        s=fit_score(cols, tr, te, y); pf.append(s)
        # also collect oof for per-class F1
        trs=_subsample(tr,y,SUBSAMPLE_PER_CLASS)
    # recompute oof properly for per-class
    oof=np.empty(len(y),object)
    for tr,te in splitter.split(Xall,y,grp):
        if len(np.unique(y[tr]))<len(classes): continue
        trs=_subsample(tr,y,SUBSAMPLE_PER_CLASS)
        try:
            p=make_pipe(y[trs]) if 'make_pipe' in globals() else None
        except TypeError:
            p=make_pipe()
        if p is None:
            # xgboost path
            m=make_clf(); m.fit(Xall[np.ix_(trs,cols)],_le.transform(y[trs]),sample_weight=_sw(y[trs]))
            oof[te]=_le.inverse_transform(m.predict(Xall[np.ix_(te,cols)]))
        else:
            p.fit(Xall[np.ix_(trs,cols)],y[trs]); oof[te]=p.predict(Xall[np.ix_(te,cols)])
    fill=oof!=None
    mf1=f1_score(y[fill],oof[fill].astype(str),labels=classes,average=None)
    return np.mean(pf), np.std(pf), _acc(y[fill],oof[fill].astype(str)), mf1

_fmean,_fsd,_facc,_fperclass=_full_grouped_cv()
print(f'FULL-48 baseline: Acc {_facc*100:.1f}%  MacroF1 {_fmean*100:.1f}% (sd {_fsd*100:.1f})')
print(f'  per-class F1: ' + '  '.join(f'{c} {v*100:.1f}%' for c,v in zip(classes,_fperclass)))
print(f'\nnested (leakage-free) MacroF1 was {outer_mf1.mean()*100:.1f}% (sd {outer_mf1.std()*100:.1f})')
_gap=(_fmean-outer_mf1.mean())*100
print(f'FULL-48 minus nested-selection: {_gap:+.1f} pts')
if _gap > max(_fsd,outer_mf1.std())*100:
    print('READ: full-48 BEATS selection beyond fold noise -> feature selection HURTS here; use all 48 (as in 3-class).')
elif _gap < -max(_fsd,outer_mf1.std())*100:
    print('READ: selection BEATS full-48 beyond noise -> selection is worthwhile for this classifier in 2-class.')
else:
    print('READ: full-48 and selection are within fold noise -> selection neither helps nor hurts; prefer full-48 for simplicity/robustness.')
